# Chapter 3 — Fitting a model · Solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch03_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python.

**Loss used**: the mean squared error (equation 3.2) on homes A to F, unless stated otherwise.

**Contents.** Data · The computations of lessons 3.1 to 3.6 · Exercises 3.1 to 3.6

*The cell of lesson 3.4 and the cell of exercise 3.4 f) run hundreds of thousands of steps: allow a few seconds for each.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book colors: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifier D0.1 is the one used in the book and in the CSV file.

In [ ]:
# Chapter 3 data, copied from the book's CSV file (do not modify)

# D0.1 — the six homes from the chapter opening, used for fitting
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)

# D3.1 — four new sales, used only for evaluation (lesson 3.6)
new_sales = pd.DataFrame({
    'sale':      ['V1', 'V2', 'V3', 'V4'],
    'floor_area': [38, 58, 76, 92],
    'price':  [132, 215, 254, 322],
})
xv = new_sales['floor_area'].to_numpy(dtype=float)
yv = new_sales['price'].to_numpy(dtype=float)

# Home G (chapter opening): worked example, never used for fitting
x_G, y_G = 62, 209

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `sale` | identifier of the new sale (V1 to V4) | — | `vente` |

## The computations of the lessons, in Python

These cells redo the computations of lessons 3.1 to 3.6.

In [ ]:
# Lesson 3.1 — two lines, two error measures (equations 3.1 and 3.2)
def mae(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for name, a, b in [("opening", 3, 30), ('colleague', 3.3, 14)]:
    print(f'{name:10s} MAE = {mae(a, b):.2f} k€   MSE = {mse(a, b):.2f} k€²   RMSE = {np.sqrt(mse(a, b)):.2f} k€')

In [ ]:
# Lesson 3.2 — the loss curve, b fixed at 30 k€ (equation 3.3)
slopes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(mse(a, 30), 2) for a in slopes], index=slopes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [mse(a, 30) for a in A], color=BLUE)
plt.scatter(slopes, [mse(a, 30) for a in slopes], color='black', zorder=3)
plt.xlabel('Slope a (k€ per m²)'); plt.ylabel('Loss L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 3.3 — secants, derivative (equation 3.4) and two updates (equation 3.5)
def derivative_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'secant h = {h}: {(mse(2.9 + h, 30) - mse(2.9, 30)) / h:.1f}')
print('exact derivative at 2.9:', round(derivative_a(2.9), 1))

a, eta = 2.5, 0.00006
for stage in range(3):
    print(f'step {stage}: a = {a}, L = {mse(a, 30):.2f}, L\'(a) = {derivative_a(a):.2f}')
    a = round(a - eta * derivative_a(a), 3)    # rounded to the thousandth, as in the book

In [ ]:
# Lesson 3.4 — the gradient (equation 3.6) and one step on both coefficients (equation 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('gradient at (2.5, 30):', g.round(2))
print('after one step:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'after {k} steps: a = {a:.4f}, b = {b:.4f}, L = {mse(a, b):.2f}')
a_ls, b_ls = np.polyfit(x, y, 1)
print(f'bottom of the valley: a = {a_ls:.3f}, b = {b_ls:.2f}, L = {mse(a_ls, b_ls):.2f}')
print(f'G, worked example: {a_ls * x_G + b_ls:.0f} k€ predicted for {y_G} k€ observed')

In [ ]:
# Lesson 3.5 — two descents starting from the same predictions (equation 3.8)
mu, sigma = x.mean(), x.std()          # std() divides by n, as in chapter 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descent_m2(eta, n_steps, a=2.5, b=30.0):
    for _ in range(n_steps):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descent_std(eta, n_steps, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # equation 2.7: same predictions at the start
    for _ in range(n_steps):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # equation 3.8: back to (a, b)

print('steps    | m², η = 0.00006           | standardized, η = 0.3')
a, b = 2.5, 30.0
done = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descent_m2(0.00006, k - done, a, b); done = k
    a_s, b_s = descent_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {mse(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {mse(a_s, b_s):8.2f}')

# Limit learning rates: beyond them, the descent diverges (about 0.00022 in m², 1 when standardized)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: loss after 400 steps = {mse(*descent_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'standardized, η = {eta}: loss after 400 steps = {mse(*descent_std(eta, 400)):.4g} k€²')

In [ ]:
# Lesson 3.6 — least-squares line (equation 3.9) and R² (equation 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_ls = cov / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_ls:.3f} k€/m², b = {b_ls:.2f} k€')

def r2_lesson(observed, predicted, reference):
    """R² = 1 − MSE of the rule / MSE of the constant prediction “reference”."""
    return 1 - np.mean((observed - predicted) ** 2) / np.mean((observed - reference) ** 2)

y_mean = y.mean()                      # mean price of A to F: the reference for R² on A to F
print(f'A to F: constant MSE = {np.mean((y - y_mean) ** 2):.2f}, line MSE = {mse(a_ls, b_ls):.2f}, '
      f'R² = {r2_lesson(y, a_ls * x + b_ls, y_mean):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# The three lines, the same measure, on A to F then on V1 to V4 (lesson 3.6)
lines = {'least squares': (a_ls, b_ls), 'opening': (3, 30), 'colleague': (3.3, 14)}
for name, (a, b) in lines.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{name:16s} A to F: MSE = {np.mean(e_af ** 2):6.2f}  MAE = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 to V4: MSE = {np.mean(e_v ** 2):6.2f}  MAE = {np.mean(np.abs(e_v)):5.2f}   deviations {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # if V4 had sold for 312 k€
for name in ('least squares', 'colleague'):
    a, b = lines[name]
    print(f'V4 at 312 k€: {name} MSE = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, worked example: {a_ls * x_G + b_ls:.2f} k€ predicted, deviation {y_G - (a_ls * x_G + b_ls):.2f} k€')

## Exercise 3.1 — Reading a graph: two measures, two minimums

*Data D0.1. Concepts: MAE and MSE (lesson 3.1), loss curve (lesson 3.2), median (lesson 1.3).* The slope is fixed at 3 k€/m². First answer questions a) to e) by reading the figure in the book.

### a) to e) Solution

**a)** The MSE is smallest around $b \approx 30$ k€, where it is about 57 k€². The computation in f) gives exactly $b = 30.33$ and 57.22 k€².

**b)** The MAE curve is made of straight-line segments. Around its minimum, it forms a horizontal **plateau**, from $b = 24$ to $b = 35$ k€, at a height of about 7.3 k€. All these values of $b$ give the same MAE.

**c)** For $b = 30$, we read an MSE of about 57 k€² and an MAE of about 7.3 k€. The table of deviations in lesson 3.1 gives 57.33 k€² and 7.33 k€ for the opening line: the reading is consistent.

**d)** With $b = 25$, the deviations are $r_i - 25$: −3, 12, −2, 10, −1 and 16 k€. Their absolute values add up to 44, as for $b = 30$: the MAE is still 44 / 6 ≈ 7.33 k€. According to the MAE, the two lines are **equivalent**. Their squares add up to 9 + 144 + 4 + 100 + 1 + 256 = 514, hence an MSE of 514 / 6 ≈ 85.67 k€², against 57.33: according to the MSE, the colleague's line is **worse**. The verdict depends on the measure, as in lesson 3.1.

**e)** On the plateau, the slope of the curve is 0. A gradient descent that reaches it stops wherever it is: it cannot choose between 24 and 35 k€. At the corners, the slope changes abruptly and the derivative does not exist. The MSE curve is a smooth parabola: it has a single lowest point, and its slope, proportional to the distance to that point, tells at each step which direction to go and how far.

### f) In the notebook

In [ ]:
def mse_b(b):
    return np.mean((y - (3 * x + b)) ** 2)

def mae_b(b):
    return np.mean(np.abs(y - (3 * x + b)))

In [ ]:
# Check, then plot of the two curves
if mse_b(30) is None or mae_b(30) is None:
    print('Complete mse_b and mae_b.')
else:
    print('mse_b and mae_b:', 'correct' if np.isclose(mse_b(30), 344 / 6) and np.isclose(mae_b(30), 44 / 6) else 'check again')
    B = np.linspace(15, 45, 601)
    fig, (h, low) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [mse_b(b) for b in B], color=BLUE); h.set_ylabel('MSE (k€²)'); h.grid(alpha=0.4)
    low.plot(B, [mae_b(b) for b in B], color=ORANGE); low.set_ylabel('MAE (k€)'); low.grid(alpha=0.4)
    low.set_xlabel('Intercept b (k€)')
    plt.show()
    print('corners of the MAE curve (sorted r_i):', np.sort(y - 3 * x))

In [ ]:
r_sorted = np.sort(y - 3 * x)                    # 22, 23, 24, 35, 37, 41
median_r = (r_sorted[2] + r_sorted[3]) / 2       # n = 6 is even: ranks 3 and 4 → 29.5
mean_r = np.mean(y - 3 * x)                  # 182 / 6 ≈ 30.33

In [ ]:
# Check
for name, yours, truth in [('median', median_r, 29.5), ('mean', mean_r, 182 / 6)]:
    print(f'{name} of the r:', 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again'))
if median_r is not None and mae_b(30) is not None:
    print(f'MAE at the median: {mae_b(median_r):.2f} k€ (on the plateau); '
          f'MSE at the mean: {mse_b(mean_r):.2f} k€² (minimum)')

### Solution of f)

The corners are at 22, 23, 24, 35, 37 and 41 k€, the six values $r_i$. The plateau runs from the third to the fourth sorted value. The median of the $r_i$, $(24 + 35)/2 = 29.5$ k€, lies on it. The minimum of the MSE is reached at the mean of the $r_i$, 30.33 k€. This brings back chapter 1: the mean answers the MSE and the median answers the MAE. The median resists extreme values, the mean does not. Likewise, the MSE penalizes large errors more than the MAE does.

## Exercise 3.2 — Two steps of gradient descent

*Data D0.1. Concepts: mean squared error (lesson 3.1), derivative and update (lesson 3.3).* The slope is fixed at 3 k€/m². We are looking for the intercept $b$. First do questions a) to d) on paper.

### a) Prepare

In [ ]:
r = y - 3 * x
r

In [ ]:
# Check
if r is None:
    print('Complete the previous cell.')
else:
    print('r:', 'correct' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'check again')

### a) to c) Solution on paper

**a)** $r = (22, 37, 23, 35, 24, 41)$ k€. For example for A: 118 − 3 × 32 = 118 − 96 = 22. The deviation is then $e_i = y_i - 3x_i - b = r_i - b$.

**b) and c)** With $\eta = 0.3$, the update is $b - 0.3 \times \left(-\frac{2}{6}\sum e_i\right) = b + 0.1 \sum e_i$.

| Step | $b$ (k€) | Deviations $r_i - b$ (k€) | $\sum e_i$ | $L(b)$ (k€²) | $L'(b)$ | Step size |
|:--:|--:|:--|--:|--:|--:|--:|
| 0 | 20 | 2, 17, 3, 15, 4, 21 | 62 | 984 / 6 = 164 | −20.67 | +6.2 |
| 1 | 26.2 | −4.2, 10.8, −3.2, 8.8, −2.2, 14.8 | 24.8 | 445.84 / 6 ≈ 74.31 | −8.27 | +2.48 |
| 2 | 28.68 | −6.68, 8.32, −5.68, 6.32, −4.68, 12.32 | 9.92 | ≈ 59.96 | −3.31 | |

Details of step 0: the squares are 4, 289, 9, 225, 16 and 441, for a sum of 984. The derivative is $-\frac{2}{6} \times 62 \approx -20.67$. The step is $0.3 \times 20.67 = 6.2$, and $b_1 = 20 + 6.2 = 26.2$.

In [ ]:
L_b0 = 164
derivative_b0 = -20.67
b1 = 26.2
L_b1 = 74.31
b2 = 28.68
L_b2 = 59.96

In [ ]:
# Check (independent computation)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
expected_values = {'L_b0': (L_b0, _L(20)), 'derivative_b0': (derivative_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for name, (yours, truth) in expected_values.items():
    status = 'not yet completed' if yours is None else ('correct' if abs(yours - truth) < 0.01 else 'check again')
    print(f'{name}: {status}')

### d) Interpret

The loss decreases at each step: 164, then 74.31, then 59.96 k€². The steps get shorter: 6.2, then 2.48. With $\eta = 0.3$, each step reduces the distance to the minimum by 60%. Indeed, $b_{\text{new}} = b + 0.1 \times 6\,(\bar{r} - b)$, so $\bar{r} - b$ is multiplied by 0.4. The descent heads toward $\bar{r} = 182/6 \approx 30.33$ k€, the **mean of the $r_i$**. At the minimum, the loss equals the variance of the $r_i$, about 57.22 k€², that is $7.56^2$: this is the standard deviation of the deviations of the opening line (exercise 1.6).

### e) In the notebook

In [ ]:
def loss(b):
    """MSE of the line y = 3x + b on A to F (equation 3.2)."""
    e = y - (3 * x + b)
    return np.mean(e ** 2)

def derivative(b):
    """L'(b) = −(2/n) Σ e_i (equation 3.6)."""
    e = y - (3 * x + b)
    return -2 / len(x) * np.sum(e)

The two functions follow equations 3.2 and 3.6 word for word, with a slope fixed at 3.

In [ ]:
def descent(b0, eta, n_steps):
    """Returns the list of b values and the list of losses, from step 0 to step n_steps."""
    bs, losses = [b0], [loss(b0)]
    for _ in range(n_steps):
        bs.append(bs[-1] - eta * derivative(bs[-1]))   # equation 3.5
        losses.append(loss(bs[-1]))
    return bs, losses

if loss(20) is None or derivative(20) is None:
    print('Complete loss and derivative.')
else:
    bs, losses = descent(20, 0.3, 2)
    print('two steps:', np.round(bs, 3), np.round(losses, 2), '→',
          'correct' if np.allclose(bs, [20, 26.2, 28.68]) else 'check again')
    bs, losses = descent(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(losses, marker='o', color=BLUE)
    plt.xlabel('Step'); plt.ylabel('Loss (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b after 20 steps:', round(bs[-1], 3), '  mean of the r:', round(np.mean(y - 3 * x), 3))

### f) Predict, then check: $\eta = 1.2$

**Prediction.** With $\eta = 1.2$, the update multiplies the distance to the minimum by $1 - 2\eta = -1.4$. Each step overshoots the minimum and lands farther away on the other side: $b$ oscillates on either side of 30.33 while moving away from it, and the loss increases. The descent **diverges**: 20 → 44.8 → 10.08 → 58.69 → −9.36… A learning rate that is too large does not slow the descent down: it makes it fail.

In [ ]:
if loss(20) is not None and derivative(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, losses = descent(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLUE)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Step'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Worked example on G

In [ ]:
price_G = 3 * x_G + 28.68     # = 186 + 28.68 = 214.68 k€
gap_G = y_G - price_G       # = 209 − 214.68 = −5.68 k€

In [ ]:
# Check
if price_G is None or gap_G is None:
    print('Complete the previous cell.')
else:
    print('G:', 'correct' if np.isclose(price_G, 214.68) and np.isclose(gap_G, -5.68) else 'check again')

### Solution of g)

Predicted price: 3 × 62 + 28.68 = 186 + 28.68 = **214.68 k€**. Signed deviation: 209 − 214.68 = **−5.68 k€**. G was not used to fit $b$. This computation shows how to use the rule obtained after two steps, but it does not measure its quality on new sales: a single home would not allow that, and chapter 6 will deal with this question.

## Exercise 3.3 — What does "99%" mean?

*Data D0.1 and D3.1. Concepts: coefficient of determination, fitting and evaluation (lesson 3.6).* First do questions a) to f) on paper.

### a) to f) Solution

**a)** With equation 3.10 and the MSEs of lesson 3.1: opening line, $R^2 = 1 - 57.33 / 5{,}205.22 \approx 0.989$; colleague's line, $R^2 = 1 - 65.11 / 5{,}205.22 \approx 0.987$. The reference is the constant prediction equal to the mean price of A to F, 222.33 k€, and both MSEs are computed on A to F.

**b)** The denominator, the MSE of the constant prediction on A to F, is the same for all lines. So the $R^2$ is larger when the MSE is smaller. And the least-squares line is, by definition, the one whose MSE on A to F is smallest. No line can have a better $R^2$ on these six sales.

**c)** Predicted prices: 3 × 38 + 30 = 144, 3 × 58 + 30 = 204, 3 × 76 + 30 = 258 and 3 × 92 + 30 = 306 k€. Deviations: 132 − 144 = −12, 215 − 204 = +11, 254 − 258 = −4 and 322 − 306 = +16 k€. Squares: 144, 121, 16 and 256, for a sum of 537. MSE: 537 / 4 = 134.25 k€².

**d)** On A to F: least squares (45.36), opening (57.33), colleague (65.11). On V1 to V4: colleague (70.73), least squares (93.29), opening (134.25). The reversal does not prove that the colleague's line is better. The three lines are close and four sales are very few: with V4 sold for 312 k€ instead of 322, the least-squares line would be back in front (61.11 against 73.73 k€²). It only shows that the ranking obtained during fitting does not carry over as is to new sales.

**e)** For example: "On the six sales we used to draw our line, it reduces by 98.9% the mean squared error of the constant prediction equal to their mean price, 222.33 k€. On four new sales, its mean squared error is 134.25 k€², that is about 11.6 k€." Avoid the word "explains": the $R^2$ compares two errors, it does not measure a cause.

**f)** That would mean redoing on V1 to V4 what fitting did on A to F. As soon as these sales are used to choose, they are no longer new for the chosen line: its MSE on V1 to V4 in turn becomes a selection error, an optimistic one. To report an honest error, we would need other sales, never used either for fitting or for choosing. Chapter 6 will organize this split.

### g) In the notebook

In [ ]:
def sales_mse(observed, predicted):
    """observed and predicted: numpy arrays of the same length."""
    return np.mean((observed - predicted) ** 2)

In [ ]:
# Check: the ranking of lesson 3.6, then R² on A to F (reference: mean price of A to F)
if sales_mse(y, y) is None:
    print('Complete the function sales_mse.')
else:
    a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_ls = y.mean() - a_ls * x.mean()
    expected_values = {'least squares': ((a_ls, b_ls), 45.36, 93.29), 'opening': ((3, 30), 57.33, 134.25),
                'colleague': ((3.3, 14), 65.11, 70.73)}
    for name, ((a, b), att_af, att_v) in expected_values.items():
        e1, e2 = sales_mse(y, a * x + b), sales_mse(yv, a * xv + b)
        print(f'{name:16s} A to F: {e1:6.2f}   V1 to V4: {e2:6.2f} →',
              'correct' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'check again')
    r2_af = 1 - sales_mse(y, a_ls * x + b_ls) / sales_mse(y, np.full(6, y.mean()))
    print(f'R² on A to F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'correct' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'check again')

## Exercise 3.4 — A descent on standardized areas

*Data D0.1. Concepts: gradient (lesson 3.4), standardization and conversion back (lesson 3.5), least-squares line (lesson 3.6).*

### a) Solution

With $(w, c) = (56.99, 190)$: $56.99 \times 1.4915 + 190 \approx 85.00 + 190 = 275.0$ k€. With $(a, b) = (2.5, 30)$: $2.5 \times 98 + 30 = 245 + 30 = 275$ k€. Both forms give the same prediction for F, as equation 2.7 states.

### b) Mean, standard deviation and standardized scores

In [ ]:
mu = x.mean()            # 64 m²
sigma = x.std()          # 22.796 m²: np.std divides by n by default
z = (x - mu) / sigma
z.round(4)

In [ ]:
# Check
if z is None:
    print('Complete the previous cell.')
else:
    print('z-scores:', 'correct' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'check again')

### c) The gradient in the standardized form

In [ ]:
def standardized_gradient(w, c):
    e = y - (w * z + c)
    return np.array([-2 / len(x) * np.sum(z * e), -2 / len(x) * np.sum(e)])

In [ ]:
# Check at the starting point (same predictions as (2.5, 30))
if z is None or standardized_gradient(1, 1) is None:
    print('Complete cells b) and c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = standardized_gradient(w0, c0)
    print(f'start: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'correct' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'check again')

### d) Thirty steps, then back to the original units

In [ ]:
w, c = 2.5 * sigma, 30 + 2.5 * mu
for _ in range(30):
    w, c = np.array([w, c]) - 0.3 * standardized_gradient(w, c)
a_std = w / sigma
b_std = c - a_std * mu
print(f'w = {w:.3f}, c = {c:.3f}  →  a = {a_std:.3f} k€/m², b = {b_std:.2f} k€')

In [ ]:
# Check: equation 3.9 and np.polyfit
a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'equation 3.9: a = {a_ls:.3f}, b = {b_ls:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Complete the previous cell.')
else:
    print('line obtained:', 'correct' if np.allclose([a_std, b_std], [a_ls, b_ls], atol=1e-4) else 'check again')

### e) and f) The loss of the two descents, then the number of steps

In [ ]:
# The two descents of lesson 3.5, step by step (nothing to complete)
def path_m2(n_steps, eta=0.00006):
    a, b, losses = 2.5, 30.0, []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return losses

def path_std(n_steps, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, losses = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return losses

plt.figure(figsize=(6, 3))
plt.semilogy(path_m2(30), marker='o', color=ORANGE, label='areas in m², η = 0.00006')
plt.semilogy(path_std(30), marker='s', color=BLUE, label='standardized areas, η = 0.3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Number of steps'); plt.ylabel('Loss (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Number of steps to get within 0.1 k€² of the minimum (a few seconds)
L_min = np.mean((y - (a_ls * x + b_ls)) ** 2)

def n_steps(X, a, b, eta, limit=10 ** 6):
    for k in range(1, limit + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

losses_std = path_std(30)
print('standardized areas, η = 0.3:', next(k for k, v in enumerate(losses_std) if v - L_min < 0.1), 'steps')
print('areas in m², η = 0.00006:      ', n_steps(x, 2.5, 30.0, 0.00006), 'steps')

### Solution of f)

The standardized descent gets within 0.1 k€² of the minimum in **6 steps**, the descent in m² in **169,808 steps**. Both find the same line, 3.151 × area + 20.67. Key takeaway: before a gradient descent, standardize columns with very different scales. To read the coefficients in the original units, convert them back afterwards with equation 3.8.

## Exercise 3.5 — Predict, then check: areas in tens of m²

*Data D0.1. Concepts: change of scale (lesson 2.5), gradient (lesson 3.4), standardization (lesson 3.5).*

### a) to d) Expected predictions

**a)** The line does not change: these are the same homes and the same prices, only the unit of the column changes. One ten-m² unit is 10 m², so the slope is multiplied by 10: 31.51 k€ per ten m². The intercept, in k€, stays 20.67.

**b)** The starting setting becomes (25, 30). The deviations are the same, and each $x_i$ is divided by 10. The quotient is therefore 74.5 / 10 ≈ 7.45 tens of m², that is still 74.5 m²: the same area, expressed in another unit.

**c)** Larger. The derivative with respect to the slope is divided by 10, and the same move of the line requires a slope 10 times larger. The limit learning rate is therefore multiplied by about 10 × 10 = 100: it goes from about 0.00022 to about 0.021.

**d)** Fewer steps than the descent in m², because the steps can be 100 times larger. But more than the standardized descent: the areas are not centered, and the valley remains tilted.

### e) Check

In [ ]:
x_tens = x / 10
x_tens

In [ ]:
# Check, then the measurements (nothing else to complete)
if x_tens is None:
    print('Complete the previous cell.')
else:
    print('x_tens:', 'correct' if np.allclose(x_tens, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'check again')
    print('a) least-squares line (slope, intercept):', np.round(np.polyfit(x_tens, y, 1), 2))
    e0 = y - (25 * x_tens + 30)
    print(f'b) quotient at step 0: {np.sum(x_tens * e0) / np.sum(e0):.2f} tens of m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_tens - b
            a, b = a + eta * 2 / 6 * np.sum(x_tens * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: loss after 200 steps = {np.mean((y - a * x_tens - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_tens - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_tens * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_tens - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0.006: {k} steps to get within 0.1 k€² of the minimum')

### e) and f) Solution

The notebook confirms the predictions: slope 31.51 k€ per ten m², intercept 20.67 k€; quotient 7.45; with 0.021 the descent converges, with 0.022 it diverges; with $\eta = 0.006$, it takes **1,785 steps**, against 169,808 for areas in m² and 6 for standardized areas (exercise 3.4).

Changing the unit did not change the optimal line: 31.51 k€ per ten m² is 3.151 k€ per m². It changed the path and the usable learning rate. Standardization goes further. It divides by the standard deviation, which puts both coefficients on the same scale. It also removes the mean: with $\sum z_i = 0$, the slope and the intercept no longer interfere with each other, and the valley stops being tilted (exercise 3.6).

## Exercise 3.6 — Challenge: circles on the map

*Data D0.1. Concepts: standardization (lessons 1.4 and 3.5), partial derivatives and gradient (lesson 3.4).* Questions a) to f) are done on paper. The notebook checks your results.

### a) to f) Solution

**a)** $\sum z_i = \sum (x_i - \mu)/\sigma = (\sum x_i - n\mu)/\sigma = 0$, since $n\mu = \sum x_i$. And $\sum z_i^2 = \sum (x_i - \mu)^2 / \sigma^2 = n\sigma^2 / \sigma^2 = n$, since the standard deviation divides by $n$.

**b)** $\partial L / \partial c = -\frac{2}{n} \sum (y_i - w z_i - c) = -\frac{2}{n}(n\bar{y} - w \times 0 - nc) = 2(c - \bar{y})$. $\partial L / \partial w = -\frac{2}{n} \sum z_i (y_i - w z_i - c) = -\frac{2}{n}(nm - wn - c \times 0) = 2(w - m)$. Both derivatives vanish for $w^* = m$ and $c^* = \bar{y} = 222.33$ k€. Here $m = 1{,}637.5 / 22.796 \approx 71.83$ k€. Converted back with equation 3.8: $a = m / \sigma = \operatorname{cov}(x, y)/\sigma_x^2 \approx 3.151$, the slope of equation 3.9.

**c)** $w_{\text{new}} = w - \eta \times 2(w - w^*)$, so $w_{\text{new}} - w^* = (1 - 2\eta)(w - w^*)$. The computation is the same for $c$. With $\eta = 0.3$, the factor is $1 - 0.6 = 0.4$. In exercise 3.2, the derivative was already $2(b - \bar{r})$, hence the same factor.

**d)** With $\eta = 0.5$, the factor is 0: a single step reaches the minimum. The descent converges as long as $|1 - 2\eta| < 1$, that is $0 < \eta < 1$. With $\eta = 1$, the factor is −1: the setting bounces back and forth forever between two points. Beyond 1, the descent diverges.

**e)** Expanding the square and using a): $L = \overline{y^2} + w^2 + c^2 - 2wm - 2c\bar{y} = (w - m)^2 + (c - \bar{y})^2 + \overline{y^2} - m^2 - \bar{y}^2$, where $\overline{y^2}$ is the mean of the squared prices. The last term is the value of $L$ at the minimum: $54{,}637.33 - 5{,}159.86 - 49{,}432.11 \approx 45.36$ k€². A contour line gathers the settings with the same loss, hence with the same distance $(w - w^*)^2 + (c - c^*)^2$ to the minimum: it is a circle centered on the minimum. The gradient points toward the center, hence the almost straight path of lesson 3.5.

**f)** In m², $\sum x_i = 384 \neq 0$ and $\frac{1}{n}\sum x_i^2 \approx 4{,}615.67 \neq 1$. The expansion of $L(a, b)$ then contains a cross term $2 \times 64\, ab$, and $a^2$ weighs about 4,600 times more than $b^2$. The contour lines are very elongated, tilted ellipses: this is the valley in the book's figure.

### Check in the notebook

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) sum of the z = {np.sum(zz):.1e}, sum of the z² = {np.sum(zz ** 2):.4f}')

In [ ]:
m = np.mean(zz * y)

def step(w, c, eta):
    """Returns (w, c) after one update (equation 3.7 with z instead of x)."""
    e = y - (w * zz + c)
    return w + eta * 2 / len(x) * np.sum(zz * e), c + eta * 2 / len(x) * np.sum(e)

In [ ]:
# Check of b) and d)
if m is None or step(1, 1, 0.1) is None:
    print('Complete the previous cell.')
else:
    print(f'b) m = {m:.2f} →', 'correct' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'check again')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = step(w0, c0, 0.5)
    print(f'd) one step with η = 0.5: (w, c) = ({w1:.2f}, {c1:.2f}) →',
          'correct' if np.allclose([w1, c1], [m, y.mean()]) else 'check again')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        sequence = []
        for _ in range(4):
            w, c = step(w, c, eta)
            sequence.append(round(c, 2))
        print(f'   η = {eta}: c after 1, 2, 3, 4 steps = {sequence}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLUE)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ per standard deviation)'); plt.ylabel('c (k€)'); plt.title('e) Circles'); plt.show()